#  **Preprocessing**

In [0]:
!pip install kagglehub[pandas-datasets]

In [0]:
import pandas as pd

# Import our synthetic dataset from GitHub repo
raw_url = "https://raw.githubusercontent.com/jazz1416/VetTrack-Customer-Service-Agent/refs/heads/main/synthetic_customer_support_tickets.csv"
df = pd.read_csv(raw_url)

print("First 5 records:", df.head())

In [0]:
# Fill NA values for text
df['Resolution'] = df['Resolution'].fillna("No resolution provided")

# Fill empty customer satisfaction ratings with a neutral median
df['Customer Satisfaction Rating'] = df['Customer Satisfaction Rating'].fillna(3)

# Cast Minutes to resolve to a nullable integer
df['Minutes to resolve'] = df['Minutes to resolve'].astype('Int64')

In [0]:
import re

# Text normalization 
def normalize_text(text):
    text = str(text).lower() # str() prevents errors on nulls
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df['Ticket Description'] = df['Ticket Description'].apply(normalize_text)
df['Resolution'] = df['Resolution'].apply(normalize_text)

# Filter the dataframe to ONLY include closed tickets for the Knowledge Base
df = df[df['Ticket Status'] == 'Closed'].reset_index(drop=True)

print(f"Filtered down to {len(df)} closed tickets for training.")

In [0]:
# Columns that should be categorized
cat_cols = ['Ticket Type', 'Ticket Subject', 'Ticket Status', 'Ticket Priority']

# Create a new column with int values for each category 
for col in cat_cols:
    numerical_values, _ = pd.factorize(df[col])
    current_index = df.columns.get_loc(col)
    new_col_name = f'{col}_id'
    df.insert(current_index + 1, column = new_col_name, value = numerical_values)

In [0]:
pip install sentence-transformers

In [0]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Map to the new description column
texts = df['Ticket Description'].tolist()

batch_size = 1000
embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True)

df['embeddings'] = list(embeddings)
print(f"Created embeddings with shape: {embeddings.shape}")

# Exploratory Data **Analysis**

### Dataset Overview

In [0]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [0]:
#column types
df.info()

In [0]:
#summary stats
df.describe()

### Feature Analysis

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

categorical_cols = ['Ticket Type', 'Ticket Subject', 'Ticket Status', 'Ticket Priority']

for col in categorical_cols:
    plt.figure(figsize=(8,4))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()


### Text analysis

In [0]:
%pip install nltk

In [0]:
from collections import Counter
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# Combine all issue descriptions
all_text = " ".join(df['Ticket Description'])

# Tokenize
words = [w for w in all_text.split() if w not in stop_words]

# Most common words
word_freq = Counter(words).most_common(20)

word_df = pd.DataFrame(word_freq, columns=['word', 'count'])
plt.figure(figsize=(10,5))
plt.bar(word_df['word'], word_df['count'])
plt.xticks(rotation=45)
plt.title("Top 20 Most Common Words in Ticket Descriptions")
plt.show()

### Embedding

In [0]:
from sklearn.decomposition import PCA

# Convert embeddings column to array
emb_matrix = np.vstack(df['embeddings'].values)

# Reduce to 2D
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(emb_matrix)

# Plot
plt.figure(figsize=(10,7))
plt.scatter(emb_2d[:,0], emb_2d[:,1], s=5, alpha=0.5)
plt.title("PCA Visualization of Ticket Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


In [0]:
from sklearn.cluster import KMeans

pca = PCA(n_components=10).fit_transform(emb_matrix)

# Cluster embeddings
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(pca)

# Visualize clusters
plt.figure(figsize=(10,7))
sns.scatterplot(x=pca[:,0], y=pca[:,1], hue=df['cluster'], palette='tab10', s=10)
plt.title("Ticket Embedding Clusters")
plt.show()

def top_words_for_cluster(cluster_id, n=20):
    texts = df[df['cluster'] == cluster_id]['Ticket Description']
    all_words = " ".join(texts).split()
    common = Counter(all_words).most_common(n)
    return common

for c in sorted(df['cluster'].unique()):
    print(f"\nCluster {c} top words:")
    print(top_words_for_cluster(c))

def sample_tickets(cluster_id, n=5):
    return df[df['cluster'] == cluster_id]['Ticket Description'].head(n)

for c in sorted(df['cluster'].unique()):
    print(f"\nCluster {c} sample tickets:")
    print(sample_tickets(c))

In [0]:
# 1. Calculate High Priority Rate
cluster_high_priority = (
    df.groupby('cluster')
      .apply(lambda x: (x['Ticket Priority'].isin(['High', 'Critical'])).mean(), include_groups=False)
      .sort_values()
)
print("High Priority Rate by Cluster:")
print(cluster_high_priority)

# 2. Utilize the new Minutes to resolve column!
cluster_resolution = df.groupby('cluster')['Minutes to resolve'].mean().sort_values()
print("\nMean Minutes to Resolve by Cluster:")
print(cluster_resolution)